# CEG-WM 端到端可执行验证 Notebook

## 目标

本 Notebook 用于生成
的可审计证据包，覆盖：

1. **模型加载**：Stable Diffusion 3.5 Medium（Transformer/DiT 架构，非 UNet）
2. **真实运行**：embed + detect 真实推理流程
3. **对齐验证**：paper_faithfulness 对齐报告必须 PASS
4. **测试验证**：pytest 全量测试通过
5. **门禁审计**：严格审计（--strict）必须 ALLOW_FREEZE
6. **证据打包**：可下载的完整 run_root + 审计报告 + 测试结果

## 输入要求

1. **项目 ZIP**：将整个 CEG-WM 仓库打包为 ZIP 上传
2. **Hugging Face Token**（可选）：若模型需要授权访问，请在下方 Cell 配置

## 输出产物

- `sd3_paper_faithfulness_run_bundle.zip`：包含 run_root、records、审计报告、测试日志
- `alignment_acceptance_summary.json`：对齐验收摘要

## A. 运行前准备：上传项目 ZIP

In [ ]:
# 克隆仓库到 Colab 环境
import os

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "Content_Chain"  # 包含 supported_models 声明的分支
REPO_NAME = "CEG-WM"
WORK_DIR = f"/content/{REPO_NAME}"

# 先切换到安全的目录（/content）
os.chdir("/content")
print(f"切换到临时目录: {os.getcwd()}")

# 如果目录已存在，先删除
if os.path.exists(WORK_DIR):
  print(f"清空已存在的目录: {WORK_DIR}")
  import shutil
  shutil.rmtree(WORK_DIR)
  print("✅ 已清空")

# 克隆仓库（从 Content_Chain 分支）
print(f"\n克隆仓库: {REPO_URL} (分支: {REPO_BRANCH})")
!git clone -b {REPO_BRANCH} {REPO_URL}

# 进入工作目录
os.chdir(WORK_DIR)
print(f"\n当前工作目录: {os.getcwd()}")
print(f"✅ 仓库克隆完成: {WORK_DIR} (源于分支: {REPO_BRANCH})")

## B. 安装依赖与环境固定

In [ ]:
# Cell B: 安装依赖
from pathlib import Path

print("=" * 60)
print("[B] 安装依赖包")
print("=" * 60)

# 检查 requirements.txt 是否存在
req_file = Path("requirements.txt")
if not req_file.exists():
    raise RuntimeError("未找到 requirements.txt 文件")

print("\n正在安装依赖（这可能需要几分钟）...")

# 安装核心依赖（使用仓库锁定版本）
!pip install -q --upgrade pip
!pip install -q torch==2.10.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers==0.32.0 transformers==4.45.2 accelerate==1.12.0
!pip install -q safetensors==0.7.0 pyyaml==6.0.2 pillow==11.2.1
!pip install -q numpy scipy pytest pydantic
!pip install -q huggingface_hub

print("\n✅ 依赖安装完成")

# 验证关键包版本
print("\n关键包版本:")
import torch
print(f"  - PyTorch: {torch.__version__}")
print(f"  - CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - CUDA 版本: {torch.version.cuda}")
    print(f"  - GPU 设备: {torch.cuda.get_device_name(0)}")

try:
    import diffusers
    print(f"  - Diffusers: {diffusers.__version__}")
except ImportError:
    print("  - Diffusers: ❌ 未安装")

try:
    import transformers
    print(f"  - Transformers: {transformers.__version__}")
except ImportError:
    print("  - Transformers: ❌ 未安装")

try:
    import safetensors
    print(f"  - Safetensors: {safetensors.__version__}")
except ImportError:
    print("  - Safetensors: ❌ 未安装")

## C. 配置 Hugging Face 凭据（可选）

In [ ]:
# 配置 HF_TONKEN

v = os.environ.get("HF_TOKEN")
print("HF_TOKEN exists:", v is not None)
print("HF_TOKEN length:", 0 if v is None else len(v))
print("HF_TOKEN head:", "" if v is None else v[:4] + "..." + v[-4:])

# 下载 Stable Diffusion 模型
print("="*80)
print("下载 Stable Diffusion 模型")
print("="*80)

from diffusers import DiffusionPipeline
from huggingface_hub import scan_cache_dir
import torch

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

# 检查模型是否已缓存
def checkModelCached(modelId):
    try:
        cacheInfo = scan_cache_dir()
        for repo in cacheInfo.repos:
            if modelId in repo.repo_id:
                return True
        return False
    except:
        return False

if checkModelCached(MODEL_ID):
    print(f"模型已缓存: {MODEL_ID}")
    print("跳过下载")
else:
    print(f"模型未缓存 开始下载: {MODEL_ID}")
    print("这可能需要几分钟时间...")

    try:
        # 下载模型但不加载到内存
        _ = DiffusionPipeline.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )
        print("模型下载完成")
    except Exception as e:
        print(f"模型下载失败: {e}")
        print("尝试继续执行...")

# 清理内存
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("模型准备完成")

## D. 仓库自检

In [ ]:
# Cell D: 仓库自检
import sys

print("=" * 60)
print("[D] 仓库自检")
print("=" * 60)

# 语法检查
print("\n[1/3] 编译检查 main/ 模块...")
!python -m compileall main/ 2>&1 | tail -5

# 导入检查
print("\n[2/3] 测试导入核心模块...")
try:
    sys.path.insert(0, os.getcwd())
    import main
    from main.core import config_loader, digests, records_io
    from main.diffusion.sd3 import pipeline_factory
    print("✅ 核心模块导入成功")
except Exception as e:
    print(f"❌ 导入失败: {e}")
    raise

# 打印仓库版本信息
print("\n[3/3] 仓库冻结契约版本:")
try:
    from main.core.contracts import load_frozen_contracts
    contracts = load_frozen_contracts("configs/frozen_contracts.yaml")
    print(f"  - contract_version: {contracts.contract_version}")
    print(f"  - frozen_date: {contracts.frozen_date}")
    print(f"  - phase: {contracts.phase}")
except Exception as e:
    print(f"⚠️  无法读取冻结契约: {e}")

print("\n✅ 仓库自检完成")

## E. 准备运行配置

In [ ]:
# Cell E: 准备最小真实运行配置
from pathlib import Path
import json
import yaml

print("=" * 60)
print("[E] 准备运行配置")
print("=" * 60)

# ==================== 可修改参数区 ====================
# 用户可以调整以下参数

# 模型配置
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
MODEL_SOURCE = "hf"  # hf / local
HF_REVISION = "main"  # 或具体 commit hash

# 推理参数
INFERENCE_PROMPT = "a serene mountain landscape with a lake"
INFERENCE_NUM_STEPS = 20  # SD3 推荐 20-28 步
INFERENCE_GUIDANCE_SCALE = 4.5  # SD3 推荐 4.5-7.0
SEED = 42

# 水印配置
ENABLE_CONTENT = True  # 启用内容链
ENABLE_GEOMETRY = False  # 几何链（当前关闭）
ENABLE_PAPER_FAITHFULNESS = True  # 启用 paper faithfulness 对齐检查

# ⭐ P1/P2 核心配置（解决 NA 问题）
ENABLE_PIPELINE_INFERENCE = True  # ✅ 启用 Pipeline 推理（P1: pipeline_fingerprint_presence）
ENABLE_TRAJECTORY_TAP = True  # ✅ 启用轨迹采样（P2: trajectory_digest_reproducibility）

# 输出配置
RUN_ROOT = "/content/run_root"
LOGS_DIR = "/content/run_logs"

# ====================================================

# 创建输出目录
Path(RUN_ROOT).mkdir(parents=True, exist_ok=True)
Path(LOGS_DIR).mkdir(parents=True, exist_ok=True)

# 生成临时配置文件（基于 example_lf_hf_config.yaml）
# CEG-WM 不允许通过 CLI override 修改大部分参数，必须通过配置文件定义
print("\n构建临时配置文件...")

runtime_config = {
    # ==============================
    # 关键配置：启用 pipeline 构建
    # 这是生成 paper_faithfulness 证据的必要条件
    # 模型参数必须在顶层（不能在 model.* 内）
    # ==============================
    "pipeline_build_enabled": True,
    
    # ⭐ P1 关键配置：启用真实推理（替代默认的 inference_enabled=false）
    "inference_enabled": ENABLE_PIPELINE_INFERENCE,
    
    # ⭐ P2 关键配置：启用轨迹采样
    "trajectory_tap": {
        "enabled": ENABLE_TRAJECTORY_TAP,
        # 轨迹采样参数（可选，保持默认）
        "sample_at_steps": [5, 10, 15, 20],  # 采样位置
        "sample_layer_names": ["transformer"],  # 采样层
    },
    
    # 模型参数（必须在顶层以供 pipeline_factory 读取）
    "model_id": MODEL_ID,
    "model_source": MODEL_SOURCE,
    "hf_revision": HF_REVISION,
    
    "policy_path": "content_only",
    "target_fpr": 0.01,
    
    "detect": {
        "content": {
            "enabled": ENABLE_CONTENT
        },
        "geometry": {
            "enabled": ENABLE_GEOMETRY
        }
    },
    "watermark": {
        "key_id": "master_key_001",
        "pattern_id": "pattern_standard_v1",
        "strength": 0.8,
        "plan_digest": None,
        # ===== LF 链（PRC）：低频水印 =====
        "lf": {
            "enabled": True,
            "codebook_id": "lf_codebook_v1",
            "ecc": "sparse_ldpc",  # 修改为稀疏 LDPC（对应 sparse_parity_check）
            "ecc_sparsity": 3,  # 正确的参数名
            "strength": 0.5,
            "variance": 1.5  # PRC 伪高斯方差（必须是 1.5）
        },
        # ===== HF 链（T2SMark）：高频水印 =====
        "hf": {
            "enabled": True,  # 必须启用（paper spec 要求）
            "codebook_id": "hf_codebook_v1",
            "ecc": 2,
            "tail_truncation_ratio": 0.1,
            "tail_truncation_mode": "top_k_per_latent",  # 修改为 top_k_per_latent（paper spec 要求）
            "tau": 1.0  # T2SMark threshold parameter
        },
        # ===== 子空间链（Shallow Diffuse）=====
        "subspace": {
            "enabled": True,
            "frame": "latent",
            "selector_id": "selector_dct_v1",
            "grid_rows": 4,
            "grid_cols": 4,
            "grid_h": 64,
            "grid_w": 64,
            "k": 512,
            "topk": 128,
            "score_type": "amplitude",
            "channel_agg": "sum",
            "rank": 8,
            "energy_ratio": 0.9,
            "mask_digest_binding": True
        },
        "mask": {
            "threshold": 0.5,
            "resolution_binding": "512x512",
            "postprocess": {
                "enabled": True,
                "kernel": "gaussian"
            }
        },
        # ===== 注入点规范（必须定义以通过 paper_spec 对齐检查）=====
        "injection_site_spec": {
            "hook_type": "callback_on_step_end",
            "target_module_name": "StableDiffusion3Pipeline",
            "target_tensor_name": "latents",
            "hook_timing": "after_scheduler_step",
            "injection_rule_digest": "placeholder"  # 将由 CEG-WM 动态计算
        }
    },
    "generation": {
        "num_inference_steps": INFERENCE_NUM_STEPS,
        "guidance_scale": INFERENCE_GUIDANCE_SCALE,
        "prompt": INFERENCE_PROMPT,
        "seed": SEED
    },
    "model": {
        "height": 512,
        "width": 512,
        "dtype": "float16"
    },
    "enable_xformers": True,
    "evaluate": {
        "decoder_type": "content_correlation",
        "target_fpr": 0.01
    },
    "impl": {
        "content_extractor_id": "content_baseline_noop_v1",
        "geometry_extractor_id": "geometry_baseline_noop_v1",
        "fusion_rule_id": "fusion_baseline_identity_v1",
        "subspace_planner_id": "subspace_baseline_full_v1",
        "sync_module_id": "geometry_sync_baseline_v1"
    }
}

# 如果启用 paper faithfulness，在配置中标记
if ENABLE_PAPER_FAITHFULNESS:
    runtime_config["paper_faithfulness"] = {
        "enabled": True,
        "alignment_check": True
    }

# 保存临时配置文件
TEMP_CONFIG = Path("/content/temp_runtime.yaml")
with open(TEMP_CONFIG, "w") as f:
    yaml.dump(runtime_config, f, default_flow_style=False, allow_unicode=True)

print(f"✅ 临时配置文件已生成: {TEMP_CONFIG}")

print("\n配置摘要:")
print(f"  - 模型: {MODEL_ID}")
print(f"  - 模型源: {MODEL_SOURCE}")
print(f"  - 推理步数: {INFERENCE_NUM_STEPS}")
print(f"  - 引导系数: {INFERENCE_GUIDANCE_SCALE}")
print(f"  - 随机种子: {SEED}")
print(f"  - 提示词: {INFERENCE_PROMPT}")
print(f"  - Paper Faithfulness: {ENABLE_PAPER_FAITHFULNESS}")
print(f"  - Pipeline 构建: True")
print(f"  - 输出目录: {RUN_ROOT}")

# ⭐ P1/P2 配置状态
print(f"\n⭐ Paper Faithfulness 证据配置:")
print(f"  [P1] Pipeline 推理启用: {ENABLE_PIPELINE_INFERENCE}")
print(f"       → pipeline_fingerprint.status 将为 'ok'")
print(f"       → pipeline_fingerprint_presence 将为 'PASS'")
print(f"  [P2] 轨迹采样启用: {ENABLE_TRAJECTORY_TAP}")
print(f"       → trajectory_evidence.status 将为 'ok'")
print(f"       → trajectory_digest_reproducibility 将为 'PASS'")

print(f"\n水印配置:")
print(f"  - LF（PRC）启用: True")
print(f"    - ECC: sparse_ldpc")
print(f"    - Variance: 1.5")
print(f"  - HF（T2SMark）启用: True")
print(f"    - TailTruncation Mode: top_k_per_latent")
print(f"  - 子空间（Shallow Diffuse）启用: True")
print(f"  - 注入点规范已定义")

print("\n✅ 配置准备完成")

## F. 执行 Embed（真实 SD3 推理）

In [ ]:
# Cell F: 执行 embed
import subprocess
import json
import shutil
from pathlib import Path

print("=" * 60)
print("[F] 执行 Embed（真实 SD3 推理）")
print("=" * 60)

# 清空 run_root 目录（如果已存在），确保干净的执行环境
# CEG-WM 的 path_policy 要求 run_root 要么不存在，要么显式允许重用
# 因为我们每次都要做完整的新运行，直接删除并重建最简单
if Path(RUN_ROOT).exists():
    print(f"\n清空已存在的 run_root 目录: {RUN_ROOT}")
    shutil.rmtree(RUN_ROOT)
    print("✅ 已清空")

# 重新创建输出目录
Path(RUN_ROOT).mkdir(parents=True, exist_ok=True)

# 构建命令
# CEG-WM 运行时白名单仅允许有限的 override：
# - enable_geometry / disable_geometry
# - run_root_reuse_allowed / run_root_reuse_reason
# 所有模型和推理参数必须通过配置文件传递
cmd = [
    "python", "-m", "main.cli.run_embed",
    "--out", RUN_ROOT,
    "--config", str(TEMP_CONFIG)  # 使用临时生成的配置文件
]

# 仅添加被允许的 override（如果需要）
# 当前不需要任何 override，配置文件已包含所有必要参数

print("\n执行命令:")
print(" ".join(cmd))

# 执行并捕获输出
embed_log = Path(LOGS_DIR) / "embed.log"
print(f"\n日志输出: {embed_log}")
print("\n开始执行（这可能需要几分钟，首次运行会下载模型）...\n")

with open(embed_log, "w") as f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
    
    proc.wait()

if proc.returncode != 0:
    print(f"\n❌ Embed 失败，退出码: {proc.returncode}")
    print("\n最后 50 行日志:")
    !tail -50 {embed_log}
    raise RuntimeError("Embed 执行失败")

print("\n✅ Embed 执行成功")

# 定位 embed_record.json
records_dir = Path(RUN_ROOT) / "records"
embed_record_files = list(records_dir.glob("embed_*.json"))

if not embed_record_files:
    print("\n⚠️  未找到 embed_record.json，尝试查看 records/ 目录:")
    !ls -lh {records_dir}
    embed_record_path = None
else:
    embed_record_path = embed_record_files[0]
    print(f"\n找到 embed record: {embed_record_path.name}")
    
    # 抽取关键字段
    with open(embed_record_path, "r") as f:
        embed_record = json.load(f)
    
    print("\n=== Embed Record 关键字段 ===")
    
    # Paper faithfulness
    pf = embed_record.get("paper_faithfulness", {})
    if pf:
        print(f"\n[Paper Faithfulness]")
        print(f"  - spec_digest: {pf.get('spec_digest', 'N/A')[:16]}...")
    
    # Content evidence
    content_ev = embed_record.get("content_evidence", {})
    if content_ev:
        print(f"\n[Content Evidence]")
        alignment_report = content_ev.get("alignment_report", {})
        overall_status = alignment_report.get("overall_status", "UNKNOWN")
        print(f"  - alignment_report.overall_status: {overall_status}")
        print(f"  - alignment_digest: {content_ev.get('alignment_digest', 'N/A')[:16]}...")
        print(f"  - pipeline_fingerprint_digest: {content_ev.get('pipeline_fingerprint_digest', 'N/A')[:16]}...")
        print(f"  - injection_site_digest: {content_ev.get('injection_site_digest', 'N/A')[:16]}...")
        
        # 检查对齐状态
        if overall_status != "PASS":
            print(f"\n⚠️  对齐验证未通过: {overall_status}")
            failures = alignment_report.get("failures", [])
            if failures:
                print(f"\n失败原因 ({len(failures)} 项):")
                for i, fail in enumerate(failures, 1):
                    print(f"\n  [{i}] {fail.get('path', 'N/A')}")
                    print(f"      原因: {fail.get('reason', 'N/A')}")
                    if fail.get('expected'):
                        print(f"      期望: {fail.get('expected')}")
                    if fail.get('actual'):
                        print(f"      实际: {fail.get('actual')}")
        else:
            print(f"\n✅ 对齐验证通过 (overall_status=PASS)")
    
    # 输出图像
    artifacts_dir = Path(RUN_ROOT) / "artifacts"
    image_files = list(artifacts_dir.glob("*.png")) + list(artifacts_dir.glob("*.jpg"))
    if image_files:
        print(f"\n生成图像: {image_files[0].name}")
        
        # 显示图像
        from IPython.display import Image, display
        display(Image(filename=str(image_files[0])))

In [ ]:
# Cell F-debug: 详细分析对齐报告
import json
from pathlib import Path

print("=" * 60)
print("[F-Debug] 详细分析对齐报告")
print("=" * 60)

embed_record_path = Path(RUN_ROOT) / "records" / "embed_record.json"

if not embed_record_path.exists():
    print(f"\n❌ embed_record.json 不存在: {embed_record_path}")
else:
    with open(embed_record_path, "r") as f:
        embed_record = json.load(f)
    
    # 提取对齐报告
    content_ev = embed_record.get("content_evidence", {})
    alignment_report = content_ev.get("alignment_report", {})
    
    print("\n[完整对齐报告]")
    print(json.dumps(alignment_report, indent=2, ensure_ascii=False))
    
    # 详细分析
    failures = alignment_report.get("failures", [])
    passes = alignment_report.get("passes", [])
    
    print(f"\n[统计]")
    print(f"  - 通过项数: {len(passes)}")
    print(f"  - 失败项数: {len(failures)}")
    
    if failures:
        print(f"\n[失败详情]")
        for i, fail in enumerate(failures, 1):
            print(f"\n  [{i}] {fail.get('path', 'unknown')}")
            print(f"      原因: {fail.get('reason', 'N/A')}")
            if fail.get('expected') is not None:
                print(f"      期望值: {fail.get('expected')}")
            if fail.get('actual') is not None:
                print(f"      实际值: {fail.get('actual')}")
    
    if passes:
        print(f"\n[通过的检查]")
        for i, p in enumerate(passes[:5], 1):  # 只显示前 5 个
            print(f"  [{i}] {p.get('path', 'unknown')}: {p.get('reason', 'OK')}")
        if len(passes) > 5:
            print(f"  ... 还有 {len(passes) - 5} 项通过")
    
    # 检查关键字段
    print(f"\n[关键字段状态]")
    print(f"  - pipeline_fingerprint_digest: {content_ev.get('pipeline_fingerprint_digest', '<absent>')}")
    print(f"  - injection_site_digest: {content_ev.get('injection_site_digest', '<absent>')}")
    print(f"  - alignment_digest: {content_ev.get('alignment_digest', '<absent>')}")


In [ ]:
# Cell F-P1P2：验证 P1/P2 核心问题解决状态
import json
from pathlib import Path

print("=" * 80)
print("[F-P1P2] P1/P2 核心对齐检查（NA → PASS 转变验证）")
print("=" * 80)

embed_record_path = Path(RUN_ROOT) / "records" / "embed_record.json"

if not embed_record_path.exists():
    print(f"\n❌ embed_record.json 不存在: {embed_record_path}")
else:
    with open(embed_record_path, "r") as f:
        embed_record = json.load(f)
    
    content_ev = embed_record.get("content_evidence", {})
    alignment_report = content_ev.get("alignment_report", {})
    checks = alignment_report.get("checks", [])
    
    print("\n" + "=" * 80)
    print("【问题 P1】Pipeline Fingerprint Presence（管道指纹）")
    print("=" * 80)
    
    # 查找 P1 检查
    p1_check = next((c for c in checks if c.get("check_name") == "pipeline_fingerprint_presence"), None)
    
    if p1_check:
        result = p1_check.get("result")
        reason = p1_check.get("reason", "N/A")
        
        print(f"\n检查结果: {result}")
        print(f"原因: {reason}")
        
        # 检查 fingerprint 状态
        pfp = content_ev.get("pipeline_fingerprint", {})
        pfp_status = pfp.get("status")
        pfp_reason = pfp.get("reason")
        
        print(f"\n[Pipeline Fingerprint 证据状态]")
        print(f"  - status: {pfp_status}")
        print(f"  - reason: {pfp_reason}")
        
        if result == "PASS":
            print(f"\n✅ P1 PASS：Pipeline 已成功构建，指纹证据完整")
            print(f"  • inference_enabled=true 运行成功")
            print(f"  • SD3 pipeline 对象已注入 inspector")
            print(f"  • 9字段指纹已提取：{content_ev.get('pipeline_fingerprint_digest', 'N/A')[:16]}...")
        elif result == "NA":
            print(f"\n⚠️  P1 NA：Pipeline 指纹证据缺失（预期在配置启用后改为 PASS）")
            print(f"  • 原因：{pfp_reason}")
            print(f"  • 当前配置: inference_enabled={ENABLE_PIPELINE_INFERENCE}")
            if not ENABLE_PIPELINE_INFERENCE:
                print(f"  • 🔧 解决：确保配置中 inference_enabled=true")
        else:
            print(f"\n❌ P1 FAIL：{reason}")
    else:
        print(f"\n⚠️  未找到 pipeline_fingerprint_presence 检查")
    
    print("\n" + "=" * 80)
    print("【问题 P2】Trajectory Digest Reproducibility（轨迹摘要）")
    print("=" * 80)
    
    # 查找 P2 检查
    p2_check = next((c for c in checks if c.get("check_name") == "trajectory_digest_reproducibility"), None)
    
    if p2_check:
        result = p2_check.get("result")
        reason = p2_check.get("reason", "N/A")
        
        print(f"\n检查结果: {result}")
        print(f"原因: {reason}")
        
        # 检查 trajectory 状态
        traj_ev = content_ev.get("trajectory_evidence", {})
        traj_status = traj_ev.get("status")
        traj_reason = traj_ev.get("trajectory_absent_reason")
        
        print(f"\n[Trajectory Evidence 证据状态]")
        print(f"  - status: {traj_status}")
        print(f"  - absence_reason: {traj_reason}")
        
        if result == "PASS":
            print(f"\n✅ P2 PASS：轨迹采样已启用，摘要证据完整")
            print(f"  • trajectory_tap.enabled=true 运行成功")
            print(f"  • 轨迹采样点已记录")
            print(f"  • 轨迹摘要已计算：{content_ev.get('trajectory_digest', 'N/A')[:16]}...")
        elif result == "NA":
            print(f"\n⚠️  P2 NA：Trajectory 证据缺失（预期在配置启用后改为 PASS）")
            print(f"  • 原因：{traj_reason}")
            print(f"  • 当前配置: trajectory_tap.enabled={ENABLE_TRAJECTORY_TAP}")
            if not ENABLE_TRAJECTORY_TAP:
                print(f"  • 🔧 解决：确保配置中 trajectory_tap.enabled=true")
        else:
            print(f"\n❌ P2 FAIL：{reason}")
    else:
        print(f"\n⚠️  未找到 trajectory_digest_reproducibility 检查")
    
    # 整体判定（修正版：显示实际值而非假设）
    print("\n" + "=" * 80)
    print("【整体判定】Paper Faithfulness 对齐状态")
    print("=" * 80)
    
    overall = alignment_report.get("overall_status", "UNKNOWN")
    
    # 获取 P1/P2 的实际结果（不是假设）
    p1_actual = p1_check.get("result") if p1_check else "UNKNOWN"
    p2_actual = p2_check.get("result") if p2_check else "UNKNOWN"
    
    print(f"\n整体对齐状态: {overall}")
    print(f"\n关键检查结果（实际值，非推断）:")
    print(f"  • P1（Pipeline Fingerprint Presence）: {p1_actual}")
    print(f"  • P2（Trajectory Digest Reproducibility）: {p2_actual}")
    
    if p1_actual == "NA" and p2_actual == "NA" and overall == "PASS":
        print(f"\n⚠️  **重要说明**:")
        print(f"  P1/P2 都是 NA（缺失证据），但整体仍为 PASS。")
        print(f"  这是 Paper Spec 的容错设计：")
        print(f"    • 只要 P0（强制检查）通过，就容许 P1/P2 缺失")
        print(f"    • P1/P2 为 NA 时，检查被跳过而非失败")
        print(f"    • 因此整体仍可评定为 PASS")
        print(f"\n  要让 P1/P2 从 NA 变为 PASS，需要：")
        print(f"    [P1] inference_enabled=true 且管道推理成功")
        print(f"    [P2] trajectory_tap.enabled=true 且轨迹采样成功")
        print(f"\n  🔍 诊断建议：")
        print(f"    1. 在 Notebook Cell E 中检查配置是否被正确生成")
        print(f"    2. 查看 embed.log 中是否有 Pipeline/Inference 日志")
        print(f"    3. 检查 pipeline_fingerprint 的实际状态（absent/failed/ok）")
        print(f"    4. 检查 trajectory_evidence 的实际状态（absent/failed/ok）")
    elif overall == "PASS":
        print(f"\n✅ 完全对齐达成")
        if p1_actual == "PASS" and p2_actual == "PASS":
            print(f"   所有 Paper Spec 验证通过（P0+P1+P2）")
    elif overall == "PARTIAL":
        print(f"\n⚠️  整体对齐: PARTIAL（部分检查失败）")
        failed = alignment_report.get("failures", [])
        if failed:
            print(f"\n失败的检查:")
            for fail in failed:
                print(f"  - {fail.get('path', 'unknown')}: {fail.get('reason', 'N/A')}")
    else:
        print(f"\n❌ 整体对齐: {overall}")

## G. 执行 Detect

In [ ]:
# Cell G: 执行 detect
import subprocess
import json
import shutil
from pathlib import Path

print("=" * 60)
print("[G] 执行 Detect")
print("=" * 60)

if embed_record_path is None:
    print("\n⚠️  跳过 detect（embed_record 未找到）")
else:
    # 构建 detect 命令
    detect_run_root = RUN_ROOT + "_detect"
    
    # 清空 detect_run_root 目录（如果已存在），确保干净的执行环境
    # 原因：CEG-WM 的 path_policy 要求 run_root 要么不存在，要么显式允许重用
    if Path(detect_run_root).exists():
        print(f"\n清空已存在的 detect run_root 目录: {detect_run_root}")
        shutil.rmtree(detect_run_root)
        print("✅ 已清空")
    
    # 创建输出目录
    Path(detect_run_root).mkdir(parents=True, exist_ok=True)
    
    cmd = [
        "python", "-m", "main.cli.run_detect",
        "--out", detect_run_root,
        "--config", "configs/default.yaml",
        "--input", str(embed_record_path)
    ]
    
    # 注意：不要传递任何 override。detect 会从 embed_record.json 和默认配置自动继承所有参数。
    # 如果传递 override 且与配置冲突，会导致 override_value_mismatch 错误。
    
    print("\n执行命令:")
    print(" ".join(cmd[:8]) + " ...")
    
    # 执行
    detect_log = Path(LOGS_DIR) / "detect.log"
    print(f"\n日志输出: {detect_log}")
    print("\n开始执行...\n")
    
    with open(detect_log, "w") as f:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        
        for line in proc.stdout:
            print(line, end="")
            f.write(line)
        
        proc.wait()
    
    if proc.returncode != 0:
        print(f"\n❌ Detect 失败，退出码: {proc.returncode}")
        print("\n最后 50 行日志:")
        !tail -50 {detect_log}
    else:
        print("\n✅ Detect 执行成功")
        
        # 定位 detect_record.json
        detect_records_dir = Path(detect_run_root) / "records"
        detect_record_files = list(detect_records_dir.glob("detect_*.json"))
        
        if detect_record_files:
            detect_record_path = detect_record_files[0]
            print(f"\n找到 detect record: {detect_record_path.name}")
            
            with open(detect_record_path, "r") as f:
                detect_record = json.load(f)
            
            print("\n=== Detect Record 关键字段 ===")
            
            # Paper faithfulness 一致性
            pf_detect = detect_record.get("paper_faithfulness", {})
            if isinstance(pf_detect, dict):
                print(f"\n[Paper Faithfulness 一致性检查]")
                pf_status = pf_detect.get("status", "UNKNOWN")
                print(f"  - status: {pf_status}")
                
                pf_mismatch = pf_detect.get("mismatch_reasons", [])
                if pf_mismatch:
                    print(f"  - mismatch_reasons: {pf_mismatch}")
                
                if pf_status in ["mismatch", "absent", "failed"]:
                    print(f"\n⚠️  Paper faithfulness 一致性问题: {pf_status}")
                elif pf_status == "ok":
                    print(f"\n✅ Paper faithfulness 一致性验证通过")
                else:
                    print(f"\n❓ 未知状态: {pf_status}")
            else:
                print(f"\n⚠️  paper_faithfulness 字段未找到或格式错误")
            
            # 决策结果
            decision = detect_record.get("decision", {})
            if isinstance(decision, dict):
                is_watermarked = decision.get("is_watermarked")
                print(f"\n[检测决策]")
                print(f"  - is_watermarked: {is_watermarked}")
                print(f"  - decision_rule: {decision.get('rule', 'N/A')}")
            else:
                print(f"\n[检测决策]")
                print(f"  - decision 字段未找到或为空")
            
            # 执行报告
            exec_report = detect_record.get("execution_report", {})
            if exec_report:
                print(f"\n[执行报告]")
                print(f"  - content_chain_status: {exec_report.get('content_chain_status', 'N/A')}")
                print(f"  - fusion_status: {exec_report.get('fusion_status', 'N/A')}")
            
            # Record 统计
            print(f"\n[Record 统计]")
            print(f"  - 总字段数: {len(detect_record)}")
            print(f"  - 输入 record 字段数: {detect_record.get('input_record_fields', 'N/A')}")
        else:
            print("\n⚠️  未找到 detect_record.json")


In [ ]:
# Cell G-Debug: 验证 detect_record 完整结构
import json
from pathlib import Path

print("=" * 60)
print("[G-Debug] 验证 Detect Record 完整结构")
print("=" * 60)

try:
    detect_run_root = Path(RUN_ROOT + "_detect")
    detect_record_path = list((detect_run_root / "records").glob("detect_*.json"))[0]
    
    with open(detect_record_path, "r") as f:
        detect_record = json.load(f)
    
    print(f"\n✅ 已加载 detect_record.json，共 {len(detect_record)} 个字段")
    
    # 打印所有顶级字段名称
    print("\n[顶级字段列表]")
    for key in sorted(detect_record.keys()):
        value_type = type(detect_record[key]).__name__
        print(f"  - {key}: {value_type}")
    
    # 检查关键字段
    print("\n[关键字段检查]")
    print(f"  ✅ paper_faithfulness: {type(detect_record.get('paper_faithfulness'))}")
    print(f"  ✅ decision: {type(detect_record.get('decision'))}")
    print(f"  ✅ fusion_result: {type(detect_record.get('fusion_result'))}")
    print(f"  ✅ execution_report: {type(detect_record.get('execution_report'))}")
    print(f"  ✅ content_evidence_payload: {type(detect_record.get('content_evidence_payload'))}")
    
    # 详细显示 paper_faithfulness
    pf = detect_record.get("paper_faithfulness")
    if isinstance(pf, dict):
        print(f"\n[Paper Faithfulness 详情]")
        print(json.dumps(pf, indent=2, ensure_ascii=False))
    
    # 详细显示 fusion_result (前 500 字符)
    fr = detect_record.get("fusion_result")
    if fr:
        print(f"\n[Fusion Result 类型] {type(fr)}")
        if isinstance(fr, dict):
            fr_str = json.dumps(fr, indent=2, ensure_ascii=False)[:500]
            print(fr_str + ("..." if len(fr_str) > 500 else ""))
    
except Exception as e:
    print(f"\n❌ 错误: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# Cell G-Digests：摘要对照表（增强项：提高审计可读性）
import json
from pathlib import Path
import pandas as pd

print("=" * 80)
print("[G-Digests] Embed ↔ Detect 摘要对照表（完全对齐验证）")
print("=" * 80)

try:
    # 加载 embed 和 detect 记录
    embed_record_path = Path(RUN_ROOT) / "records" / "embed_record.json"
    detect_run_root = Path(RUN_ROOT + "_detect")
    detect_record_files = list((detect_run_root / "records").glob("detect_*.json"))
    
    if not embed_record_path.exists() or not detect_record_files:
        print("\n⚠️  缺少 embed 或 detect 记录，跳过对照表生成")
    else:
        with open(embed_record_path) as f:
            embed_rec = json.load(f)
        with open(detect_record_files[0]) as f:
            detect_rec = json.load(f)
        
        # 构建摘要对照表
        print("\n[对照表] 核心摘要对齐验证")
        print("-" * 80)
        
        digest_checks = []
        
        # 1. Pipeline Fingerprint Digest
        embed_pfp_digest = embed_rec.get("content_evidence", {}).get("pipeline_fingerprint_digest")
        detect_pfp_digest = detect_rec.get("content_evidence_payload", {}).get("pipeline_fingerprint_digest")
        pfp_match = "✅" if (embed_pfp_digest and detect_pfp_digest and embed_pfp_digest == detect_pfp_digest) else "❌"
        digest_checks.append({
            "摘要类型": "Pipeline Fingerprint",
            "Embed": embed_pfp_digest[:16] + "..." if embed_pfp_digest else "<absent>",
            "Detect": detect_pfp_digest[:16] + "..." if detect_pfp_digest else "<absent>",
            "对齐": pfp_match
        })
        
        # 2. Trajectory Digest
        embed_traj_digest = embed_rec.get("content_evidence", {}).get("trajectory_digest") or \
                           embed_rec.get("content_evidence", {}).get("diffusion_trace_digest")
        detect_traj_digest = detect_rec.get("content_evidence_payload", {}).get("trajectory_digest") or \
                            detect_rec.get("content_evidence_payload", {}).get("diffusion_trace_digest")
        traj_match = "✅" if (embed_traj_digest and detect_traj_digest and embed_traj_digest == detect_traj_digest) else "❌"
        digest_checks.append({
            "摘要类型": "Trajectory Digest",
            "Embed": embed_traj_digest[:16] + "..." if embed_traj_digest else "<absent>",
            "Detect": detect_traj_digest[:16] + "..." if detect_traj_digest else "<absent>",
            "对齐": traj_match
        })
        
        # 3. Alignment Report Digest
        embed_align_digest = embed_rec.get("content_evidence", {}).get("alignment_digest")
        detect_align_digest = detect_rec.get("content_evidence_payload", {}).get("alignment_digest")
        align_match = "✅" if (embed_align_digest and detect_align_digest and embed_align_digest == detect_align_digest) else "❌"
        digest_checks.append({
            "摘要类型": "Alignment Report",
            "Embed": embed_align_digest[:16] + "..." if embed_align_digest else "<absent>",
            "Detect": detect_align_digest[:16] + "..." if detect_align_digest else "<absent>",
            "对齐": align_match
        })
        
        # 4. Injection Site Digest
        embed_inj_digest = embed_rec.get("content_evidence", {}).get("injection_site_digest")
        detect_inj_digest = detect_rec.get("content_evidence_payload", {}).get("injection_site_digest")
        inj_match = "✅" if (embed_inj_digest and detect_inj_digest and embed_inj_digest == detect_inj_digest) else "❌"
        digest_checks.append({
            "摘要类型": "Injection Site Spec",
            "Embed": embed_inj_digest[:16] + "..." if embed_inj_digest else "<absent>",
            "Detect": detect_inj_digest[:16] + "..." if detect_inj_digest else "<absent>",
            "对齐": inj_match
        })
        
        # 显示为表格
        df = pd.DataFrame(digest_checks)
        print("\n" + df.to_string(index=False))
        
        # 统计对齐情况
        align_count = sum(1 for c in digest_checks if c["对齐"] == "✅")
        total_count = len(digest_checks)
        
        print(f"\n[统计] 摘要对齐: {align_count}/{total_count} 通过")
        
        if align_count == total_count:
            print(f"\n✅ 完全对齐：所有摘要在 Embed 和 Detect 之间一致，证据链完整")
        else:
            print(f"\n⚠️  部分未对齐：{total_count - align_count} 个摘要不匹配或缺失")
            print("\n详细对齐失败列表:")
            for check in digest_checks:
                if check["对齐"] == "❌":
                    print(f"  - {check['摘要类型']}")
                    print(f"    Embed: {check['Embed']}")
                    print(f"    Detect: {check['Detect']}")
        
        # 生成对照表 JSON 供后续审计使用
        checked_digests = {
            "generated_at": pd.Timestamp.now().isoformat(),
            "embed_record_path": str(embed_record_path),
            "detect_record_path": str(detect_record_files[0]),
            "checks": digest_checks,
            "alignment_summary": {
                "total": total_count,
                "passed": align_count,
                "failed": total_count - align_count,
                "complete_alignment": align_count == total_count
            }
        }
        
        # 保存到日志目录
        digest_report_path = Path(LOGS_DIR) / "checked_digests_alignment_report.json"
        with open(digest_report_path, "w") as f:
            json.dump(checked_digests, f, indent=2, ensure_ascii=False)
        
        print(f"\n✅ 摘要对照表已保存: {digest_report_path}")
        
except Exception as e:
    print(f"\n❌ 错误: {e}")
    import traceback
    traceback.print_exc()


## H. 运行 Pytest 测试

In [ ]:
# Cell H: 运行 pytest
import subprocess
import os
from pathlib import Path

# 启用 opt-in torch 测试（可选，需要 PyTorch 可用）
os.environ["CEG_WM_ENABLE_TORCH_TESTS"] = "1"

print("=" * 60)
print("[H] 运行 Pytest 测试")
print("=" * 60)
print("\n✅ 已启用 opt-in 测试: CEG_WM_ENABLE_TORCH_TESTS=1")

pytest_log = Path(LOGS_DIR) / "pytest.log"

print(f"\n日志输出: {pytest_log}")
print("\n开始执行 pytest（这可能需要几分钟）...\n")

# 执行 pytest
cmd = ["python", "-m", "pytest", "tests/", "-v", "--tb=short"]

with open(pytest_log, "w") as f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
    
    proc.wait()

if proc.returncode != 0:
    print(f"\n⚠️  Pytest 有失败用例，退出码: {proc.returncode}")
    print("\n最后 100 行日志:")
    !tail -100 {pytest_log}
else:
    print("\n✅ Pytest 全部通过")

# 解析测试结果
print("\n=== 测试结果摘要 ===")
!grep -E "passed|failed|skipped|error" {pytest_log} | tail -5 || echo "未找到摘要"

## I. 运行严格审计与冻结签署

In [ ]:
# Cell I: 运行严格审计
import subprocess
import json
from pathlib import Path

print("=" * 60)
print("[I] 运行严格审计 (--strict)")
print("=" * 60)

audits_log = Path(LOGS_DIR) / "audits_strict.log"

print(f"\n日志输出: {audits_log}")
print("\n开始执行严格审计...\n")

# 执行审计
cmd = ["python", "scripts/run_all_audits.py", "--strict"]

with open(audits_log, "w") as f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    audit_output = []
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
        audit_output.append(line)
    
    proc.wait()

if proc.returncode != 0:
    print(f"\n⚠️  审计脚本退出码: {proc.returncode}")

# 解析审计结果
print("\n=== 审计结果摘要 ===")

try:
    # 尝试从输出中提取 JSON
    audit_json_str = "".join(audit_output)
    
    # 查找 JSON 起始位置
    json_start = audit_json_str.find("{")
    if json_start >= 0:
        audit_result = json.loads(audit_json_str[json_start:])
        
        summary = audit_result.get("summary", {})
        freeze_decision = summary.get("FreezeSignoffDecision", "UNKNOWN")
        blocking_reasons = summary.get("BlockingReasons", [])
        counts = summary.get("counts", {})
        
        print(f"\n[决策]")
        print(f"  - FreezeSignoffDecision: {freeze_decision}")
        print(f"  - BlockingReasons: {blocking_reasons if blocking_reasons else '无'}")
        
        print(f"\n[统计]")
        print(f"  - PASS: {counts.get('PASS', 0)}")
        print(f"  - FAIL: {counts.get('FAIL', 0)}")
        print(f"  - BLOCK_fails: {counts.get('BLOCK_fails', 0)}")
        
        if freeze_decision == "ALLOW_FREEZE":
            print(f"\n✅ 审计通过，允许冻结 (ALLOW_FREEZE)")
        else:
            print(f"\n❌ 审计未通过: {freeze_decision}")
            
            # 打印第一条 BLOCK
            if counts.get('BLOCK_fails', 0) > 0:
                results = audit_result.get("results", [])
                for result in results:
                    if result.get("severity") == "BLOCK" and result.get("result") == "FAIL":
                        print(f"\n第一条 BLOCK 失败:")
                        print(f"  - audit_id: {result.get('audit_id', 'N/A')}")
                        print(f"  - rule: {result.get('rule', 'N/A')}")
                        print(f"  - impact: {result.get('impact', 'N/A')}")
                        break
        
        # 保存审计结果 JSON
        audit_result_path = Path(LOGS_DIR) / "audit_result.json"
        with open(audit_result_path, "w") as f:
            json.dump(audit_result, f, indent=2, ensure_ascii=False)
        print(f"\n审计结果已保存: {audit_result_path}")
        
except Exception as e:
    print(f"\n⚠️  无法解析审计结果 JSON: {e}")
    print("\n审计输出最后 50 行:")
    !tail -50 {audits_log}

## J. 生成对齐验收摘要

In [ ]:
# Cell J: 生成对齐验收摘要
import json
from pathlib import Path
from datetime import datetime

print("=" * 60)
print("[J] 生成对齐验收摘要")
print("=" * 60)

# 收集验收数据
acceptance_summary = {
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "notebook_version": "v1.0",
    "run_root": RUN_ROOT,
    "model_id": MODEL_ID,
    "configuration": {
        "seed": SEED,
        "inference_steps": INFERENCE_NUM_STEPS,
        "guidance_scale": INFERENCE_GUIDANCE_SCALE,
        "prompt": INFERENCE_PROMPT,
        "enable_paper_faithfulness": ENABLE_PAPER_FAITHFULNESS
    },
    "embed_status": "unknown",
    "detect_status": "unknown",
    "pytest_status": "unknown",
    "audit_status": "unknown",
    "final_verdict": "UNKNOWN"
}

# Embed 状态
if embed_record_path and embed_record_path.exists():
    with open(embed_record_path, "r") as f:
        embed_rec = json.load(f)
    
    content_ev = embed_rec.get("content_evidence", {})
    alignment_status = content_ev.get("alignment_report", {}).get("overall_status", "UNKNOWN")
    
    acceptance_summary["embed_status"] = {
        "success": True,
        "alignment_overall_status": alignment_status,
        "record_path": str(embed_record_path)
    }
else:
    acceptance_summary["embed_status"] = {"success": False}

# Detect 状态
try:
    detect_records_dir = Path(detect_run_root) / "records"
    detect_rec_files = list(detect_records_dir.glob("detect_*.json"))
    if detect_rec_files:
        with open(detect_rec_files[0], "r") as f:
            detect_rec = json.load(f)
        
        pf_status = detect_rec.get("paper_faithfulness", {}).get("status", "UNKNOWN")
        decision = detect_rec.get("decision", {})
        
        acceptance_summary["detect_status"] = {
            "success": True,
            "paper_faithfulness_status": pf_status,
            "is_watermarked": decision.get("is_watermarked"),
            "record_path": str(detect_rec_files[0])
        }
    else:
        acceptance_summary["detect_status"] = {"success": False}
except:
    acceptance_summary["detect_status"] = {"success": False}

# Pytest 状态
pytest_log_path = Path(LOGS_DIR) / "pytest.log"
if pytest_log_path.exists():
    with open(pytest_log_path, "r") as f:
        pytest_output = f.read()
    
    # 简单解析（寻找 "X passed" 模式）
    import re
    match = re.search(r"(\d+) passed", pytest_output)
    passed_count = int(match.group(1)) if match else 0
    
    match_failed = re.search(r"(\d+) failed", pytest_output)
    failed_count = int(match_failed.group(1)) if match_failed else 0
    
    acceptance_summary["pytest_status"] = {
        "passed": passed_count,
        "failed": failed_count,
        "all_passed": failed_count == 0
    }
else:
    acceptance_summary["pytest_status"] = {"success": False}

# Audit 状态
audit_result_path = Path(LOGS_DIR) / "audit_result.json"
if audit_result_path.exists():
    with open(audit_result_path, "r") as f:
        audit_res = json.load(f)
    
    summary = audit_res.get("summary", {})
    acceptance_summary["audit_status"] = {
        "freeze_decision": summary.get("FreezeSignoffDecision", "UNKNOWN"),
        "blocking_reasons": summary.get("BlockingReasons", []),
        "pass_count": summary.get("counts", {}).get("PASS", 0),
        "fail_count": summary.get("counts", {}).get("FAIL", 0),
        "block_fails": summary.get("counts", {}).get("BLOCK_fails", 0)
    }
else:
    acceptance_summary["audit_status"] = {"success": False}

# 最终判定
conditions = [
    acceptance_summary["audit_status"].get("freeze_decision") == "ALLOW_FREEZE",
    acceptance_summary["pytest_status"].get("all_passed", False),
    acceptance_summary["embed_status"].get("alignment_overall_status") == "PASS",
    acceptance_summary["detect_status"].get("paper_faithfulness_status") not in ["mismatch", "absent", "failed"]
]

if all(conditions):
    acceptance_summary["final_verdict"] = "ACCEPT"
    acceptance_summary["verdict_reason"] = "所有验收条件满足"
else:
    acceptance_summary["final_verdict"] = "REJECT"
    failed_conditions = []
    if not conditions[0]:
        failed_conditions.append("审计未通过 ALLOW_FREEZE")
    if not conditions[1]:
        failed_conditions.append("pytest 有失败用例")
    if not conditions[2]:
        failed_conditions.append("embed alignment_report 未通过")
    if not conditions[3]:
        failed_conditions.append("detect paper_faithfulness 一致性问题")
    acceptance_summary["verdict_reason"] = "; ".join(failed_conditions)

# 保存摘要
summary_path = Path("/content/alignment_acceptance_summary.json")
with open(summary_path, "w") as f:
    json.dump(acceptance_summary, f, indent=2, ensure_ascii=False)

print(f"\n✅ 验收摘要已生成: {summary_path}")

# 打印最终判定
print("\n" + "=" * 60)
print("最终验收判定")
print("=" * 60)
print(f"\n结果: {acceptance_summary['final_verdict']}")
print(f"原因: {acceptance_summary['verdict_reason']}")

if acceptance_summary['final_verdict'] == "ACCEPT":
    print("\n✅ 所有验收条件满足，证据包有效")
else:
    print("\n❌ 存在未满足的验收条件")

# 打印摘要
print(f"\n验收摘要详情:")
print(json.dumps(acceptance_summary, indent=2, ensure_ascii=False))

## K. 打包并下载证据包

In [ ]:
# Cell K: 打包下载
import shutil
from pathlib import Path
from google.colab import files

print("=" * 60)
print("[K] 打包证据包")
print("=" * 60)

# 创建打包目录
artifacts_out = Path("/content/artifacts_out")
artifacts_out.mkdir(parents=True, exist_ok=True)

print("\n[1/4] 复制 run_root...")
run_root_copy = artifacts_out / "run_root"
if Path(RUN_ROOT).exists():
    shutil.copytree(RUN_ROOT, run_root_copy, dirs_exist_ok=True)
    print(f"  ✅ {RUN_ROOT} → {run_root_copy}")

# 复制 detect run_root（如果存在）
try:
    if Path(detect_run_root).exists():
        detect_copy = artifacts_out / "run_root_detect"
        shutil.copytree(detect_run_root, detect_copy, dirs_exist_ok=True)
        print(f"  ✅ {detect_run_root} → {detect_copy}")
except:
    pass

print("\n[2/4] 复制 run_logs...")
logs_copy = artifacts_out / "run_logs"
if Path(LOGS_DIR).exists():
    shutil.copytree(LOGS_DIR, logs_copy, dirs_exist_ok=True)
    print(f"  ✅ {LOGS_DIR} → {logs_copy}")

print("\n[3/4] 复制验收摘要...")
summary_src = Path("/content/alignment_acceptance_summary.json")
if summary_src.exists():
    shutil.copy(summary_src, artifacts_out / "alignment_acceptance_summary.json")
    print(f"  ✅ 验收摘要已复制")

print("\n[4/4] 打包为 ZIP...")
bundle_zip = Path("/content/sd3_paper_faithfulness_run_bundle.zip")
shutil.make_archive(
    str(bundle_zip.with_suffix("")),
    'zip',
    artifacts_out
)

bundle_size_mb = bundle_zip.stat().st_size / (1024 * 1024)
print(f"  ✅ 打包完成: {bundle_zip.name} ({bundle_size_mb:.2f} MB)")

print("\n=" * 60)
print("准备下载...")
print("=" * 60)

print("\n下载文件:")
files.download(str(bundle_zip))

print("\n✅ 下载完成！")
print("\n证据包内容:")
print("  - run_root/: embed 运行完整输出（records + artifacts + logs）")
print("  - run_root_detect/: detect 运行完整输出（如果执行成功）")
print("  - run_logs/: 所有执行日志（embed/detect/pytest/audits）")
print("  - alignment_acceptance_summary.json: 验收摘要")
print("\n请解压后检查验收摘要中的 final_verdict 字段。")

## 完成

### 验收标准

检查下载的 `alignment_acceptance_summary.json` 文件，确认以下条件：

1. ✅ `audit_status.freeze_decision` == `"ALLOW_FREEZE"`
2. ✅ `pytest_status.all_passed` == `true`
3. ✅ `embed_status.alignment_overall_status` == `"PASS"`
4. ✅ `detect_status.paper_faithfulness_status` 不为 `"mismatch"`/`"absent"`/`"failed"`

若所有条件满足，`final_verdict` 应为 `"ACCEPT"`。

### 故障排查

如遇到问题：

1. **模型下载失败**：检查 Cell C 是否正确配置 HF_TOKEN
2. **Embed 失败**：查看 `run_logs/embed.log` 最后 50 行
3. **对齐未通过**：检查 embed_record.json 中 `content_evidence.alignment_report.failures`
4. **审计 BLOCK**：查看 `run_logs/audit_result.json` 中第一条 BLOCK 的详细信息

### 模型说明

本 Notebook 使用 **Stable Diffusion 3.5 Medium**，采用 **DiT (Diffusion Transformer)** 架构，而非传统 UNet。
模型通过 `diffusers` 库的 `StableDiffusion3Pipeline` 加载。